# Task4 — Full-frame EfficientNet, two-fold screening

**Run All: 4 epochs fold1 + 4 epochs fold2 แล้วหยุด**. ใช้ EfficientNet checkpoint ของแต่ละ fold, LR2e-5, weighted CE และ compound augmentation เดิม. ไม่มี MixUp; เปลี่ยน geometry จาก resize112×224+black pad เป็น resize224×224 เต็มพื้นที่ ทั้ง train และ inference

ภาพต้นฉบับไม่ได้มีข้อมูลเพิ่มจากการยืด; สมมติฐานคือขนาดวัตถุบน feature maps และการไม่เสียพื้นที่ให้ padding ช่วยได้. สัดส่วนภาพบิดและ receptive field เปลี่ยน จึงอาจแย่ลง. ต้อง fine-tune ก่อนประเมิน ไม่สลับ preprocessing ตอน Test โดยตรง

เทียบกับ baseline Public0.81047 บน OOF rows/corruption pixels เดียวกัน รวมทั้ง baseline75%+expert25%. ใช้ original/compound/single และ family-balanced gain. ไม่แก้เกณฑ์หลังเห็นผล; สองfoldเป็น screening ไม่ใช่ full validation

ประมาณ train รวม28นาทีจาก EfficientNet เดิม บวก validation. หยุด ConvNeXt ก่อน. checkpoint ทุกepoch; Run Allซ้ำโหลดcache. ไม่ใช้ Test, ไม่สร้าง submission, ไม่ฝึก folds3–5/full อัตโนมัติ

หมายเหตุ: ไม่ใช่การพิสูจน์เหตุจาก geometry ล้วน เพราะมีการฝึกเพิ่มร่วมด้วย; OOF/suite เคยใช้แล้วจึงเป็น exploratory

In [1]:
from pathlib import Path
import sys, json, hashlib, time
import numpy as np
import pandas as pd
import torch
from dataclasses import asdict
from IPython.display import display
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Task4/models/image_training.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from Task4.models import image_fullframe_training as training
from Task4.models import image_fullframe_evaluation as evaluation
from Task4.models import image_compound_validation as validation
from Task4.models.image_audit import make_folds
from Task4.models.image_compound_transforms import CONDITIONS
RAW=ROOT/'Task4/data/raw'
AUDIT=ROOT/'Task4/data/audit/followup_v2'
OUT=ROOT/'Task4/models/results/fullframe_pilot_v1'
BATCH_SIZE=32
NUM_WORKERS=0
NAME='fullframe_efficientnet4'
print('Device:',training.get_device())
if training.get_device().type=='cpu':raise RuntimeError('CPU training would be slow: select the project kernel with MPS/CUDA available.')
display(asdict(training.RECIPES[NAME]))

Device: mps


{'architecture': 'efficientnet_b0',
 'epochs': 4,
 'backbone_lr': 2e-05,
 'head_lr': 2e-05,
 'weight_decay': 0.0001,
 'schedule': 'cosine',
 'warmup_epochs': 1,
 'augmentation': 'fullframe_compound',
 'label_smoothing': 0.0,
 'seed': 42,
 'balance': 'weighted_loss',
 'jsd_weight': 0.0,
 'compound_probability': 0.6,
 'mixup_probability': 0.0,
 'mixup_alpha': 0.2}

## 1. ตรวจ Train และ matched-fold checkpoints

In [2]:
train=pd.read_csv(RAW/'train.csv')
groups=pd.read_parquet(AUDIT/'train_groups.parquet')
assert train.id.is_unique and train.label.notna().all()
assert all(np.array_equal(train[c],groups[c]) for c in ['id','file_name','label'])
metadata=pd.read_parquet(AUDIT/'images.parquet',filters=[('split','=','train')]).set_index('id').loc[train.id]
assert metadata.split.eq('train').all()
inventory=hashlib.sha256()
for row in metadata.itertuples():
    assert Path(row.file_name).name==row.file_name
    digest=validation.sha(RAW/'train'/row.file_name)
    assert digest==row.sha256, f'Changed Train image: {row.file_name}'
    inventory.update(f'{row.file_name}/{digest}'.encode())
folds=make_folds(groups,mode='near',n_splits=5,seed=42)
fold_hash=hashlib.sha256(b''.join(v.astype('int64').tobytes() for _,v in folds)).hexdigest()
assert fold_hash==json.loads((ROOT/'Task4/models/results/heavy_balance_v1/settings.json').read_text())['fold_hash']
reference_hashes={}
for fold,(fit,valid) in enumerate(folds,1):
    assert not np.intersect1d(fit,valid).size
    assert not set(groups.iloc[fit].near_group)&set(groups.iloc[valid].near_group)
    for path in evaluation.references(ROOT,fold):
        signature=json.loads((path.parent/'signature.json').read_text())
        assert signature['data_signature']==validation.sha_frame(groups.iloc[fit])
        reference_hashes[str(path.relative_to(ROOT))]=validation.sha(path)
        frame=pd.read_parquet(path.parent/'oof.parquet')
        assert frame.id.tolist()==groups.iloc[valid].id.tolist()
        assert np.array_equal(frame.true,groups.iloc[valid].label)
        reference_hashes[str((path.parent/'oof.parquet').relative_to(ROOT))]=validation.sha(path.parent/'oof.parquet')
sources={p.name:validation.sha(p) for p in (ROOT/'Task4/models').glob('image_*.py')}
settings={'recipe':asdict(training.RECIPES[NAME]),'batch_size':BATCH_SIZE,'workers':NUM_WORKERS,
          'train_manifest':validation.sha(RAW/'train.csv'),'inventory':inventory.hexdigest(),'fold_hash':fold_hash,
          'references':reference_hashes,'sources':sources,'torch':str(torch.__version__),'device':str(training.get_device())}
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'settings.json').exists():assert json.loads((OUT/'settings.json').read_text())==settings,'Settings changed: use a new OUT'
else:
    assert not any(OUT.iterdir()),'Nonempty OUT without settings'
    validation.write_json(OUT/'settings.json',settings)
print('Train verified:',len(train),'rows; Test unopened')

Train verified: 31546 rows; Test unopened


## 2. Fine-tune fold 1 — 4 epochs
checkpoint ใช้ชื่อ run ใหม่ ไม่ทับงานเดิม

In [3]:
parent=evaluation.references(ROOT,1)[2]
started=time.monotonic()
frame=training.run_fold(RAW,groups,folds,NAME,1,OUT,BATCH_SIZE,NUM_WORKERS,init_path=parent)
print('Train + original OOF minutes:',round((time.monotonic()-started)/60,1))

fold1 epoch 1 / 4 loss 0.27554 seconds 207.0
fold1 epoch 2 / 4 loss 0.18132 seconds 212.4
fold1 epoch 3 / 4 loss 0.1618 seconds 206.4
fold1 epoch 4 / 4 loss 0.14052 seconds 207.8
Train + original OOF minutes: 15.1


## 3. Paired validation และผลต่าง
Stress inference อาจนานกว่าการฝึก; reference caches โหลดกลับมาใช้เมื่อ provenance ตรงกัน

In [4]:
scores,overlap=evaluation.compare(ROOT,RAW,groups,folds,OUT,1,frame)
display(evaluation.summarize(scores))
wide=scores.pivot(index='condition',columns='procedure',values='macro_F1')
display(wide.assign(fullframe_gain=wide.fullframe-wide.baseline,blend_gain=wide.fullframe_blend25-wide.baseline))
display(overlap[overlap.condition.eq('original')])
# Family-balanced stress; don't let six compound cases dominate the mean.
s=scores.copy()
s['family']=s.condition.map(lambda c:'original' if c=='original' else ('compound' if c.startswith('compound_') else CONDITIONS[c][1]))
f=s.groupby(['procedure','family']).macro_F1.mean().unstack()
reports={}
for name in ['fullframe','fullframe_blend25']:
    delta=f.loc[name]-f.loc['baseline']
    reports[name]={'original_gain':float(delta.original),
                   'stress_family_gain':float(delta.drop('original').mean()),
                   'worth_reviewing':bool(delta.original>=-.001 and delta.drop('original').mean()>=.002)}
validation.write_json(OUT/'pilot_report.json',{'fold':1,'reports':reports,
    'automatic_continuation':False,'test_used':False,
    'note':'Single fold exploratory screening only; no claim of 5-fold improvement'})
display(reports)
print('Fold 1 complete/cached. Continue to section 4 for the authorized fold 2 experiment.')

stress fold1 gaussian_1
stress fold1 gaussian_2
stress fold1 speckle_1
stress fold1 speckle_2
stress fold1 blur_1
stress fold1 blur_2
stress fold1 jpeg_1
stress fold1 jpeg_2
stress fold1 dark_1
stress fold1 dark_2
stress fold1 compound_dark_noise
stress fold1 compound_color_noise
stress fold1 compound_rotation_dark
stress fold1 compound_bright_color
stress fold1 compound_dark_color_noise
stress fold1 compound_rotation_color_noise


suite,compound,original,single
procedure,,,
baseline,0.944725,0.986331,0.977201
fullframe,0.915630,0.974377,0.949144
fullframe_blend25,0.943747,0.986480,0.977978


procedure,baseline,fullframe,fullframe_blend25,fullframe_gain,blend_gain
condition,,,,,
blur_1,0.982007,0.956038,0.979976,-0.025969,-0.002031
blur_2,0.976005,0.897169,0.975972,-0.078835,-0.000033
compound_bright_color,0.947935,0.919854,0.949965,-0.028081,0.002030
compound_color_noise,0.949996,0.933860,0.948072,-0.016136,-0.001924
compound_dark_color_noise,0.927962,0.891893,0.931979,-0.036069,0.004018
compound_dark_noise,0.954018,0.915592,0.949905,-0.038426,-0.004114
compound_rotation_color_noise,0.926192,0.906238,0.922190,-0.019954,-0.004003
compound_rotation_dark,0.962249,0.926342,0.960369,-0.035907,-0.001880
dark_1,0.982041,0.964063,0.979998,-0.017978,-0.002044


,fold,phase,condition,rows,baseline_errors,candidate_errors,both_wrong,candidate_fixes,candidate_breaks
0,1,development,original,6310,70,137,51,19,86


{'fullframe': {'original_gain': -0.011953884653292612,
  'stress_family_gain': -0.02823022057983175,
  'worth_reviewing': False},
 'fullframe_blend25': {'original_gain': 0.00014888980502036908,
  'stress_family_gain': 0.0004840580404350536,
  'worth_reviewing': False}}

Fold 1 complete/cached. Continue to section 4 for the authorized fold 2 experiment.


## 4. Fold 2 — confirmation ก่อนลงทุน folds ที่เหลือ

ใช้ recipe และน้ำหนัก ensemble เดิมทั้งหมด ไม่ปรับตามผล fold 1. เริ่มจาก EfficientNet **fold 2** ที่ไม่เคยฝึกบน validation ของ fold 2; เพิ่ม 4 epochs พร้อม checkpoint. ถ้า kernel เดิมยังอยู่ รันตั้งแต่เซลล์นี้ได้; ถ้า restart ให้ Run All เพื่อโหลด fold 1 กลับมา

เป็นการสำรวจบน validation ที่ใช้ซ้ำ ไม่ใช่หลักฐานรับรองคะแนน Kaggle. **หยุดหลัง fold 2** ไม่ฝึก full model

In [5]:
parent2=evaluation.references(ROOT,2)[2]
started=time.monotonic()
frame2=training.run_fold(RAW,groups,folds,NAME,2,OUT,BATCH_SIZE,NUM_WORKERS,init_path=parent2)
print('Fold 2 train + original OOF minutes:',round((time.monotonic()-started)/60,1))
scores2,overlap2=evaluation.compare(ROOT,RAW,groups,folds,OUT,2,frame2)
display(evaluation.summarize(scores2))

fold2 epoch 1 / 4 loss 0.27572 seconds 206.6
fold2 epoch 2 / 4 loss 0.18414 seconds 212.8
fold2 epoch 3 / 4 loss 0.16263 seconds 212.6
fold2 epoch 4 / 4 loss 0.14922 seconds 212.5
Fold 2 train + original OOF minutes: 15.3
stress fold2 gaussian_1
stress fold2 gaussian_2
stress fold2 speckle_1
stress fold2 speckle_2
stress fold2 blur_1
stress fold2 blur_2
stress fold2 jpeg_1
stress fold2 jpeg_2
stress fold2 dark_1
stress fold2 dark_2
stress fold2 compound_dark_noise
stress fold2 compound_color_noise
stress fold2 compound_rotation_dark
stress fold2 compound_bright_color
stress fold2 compound_dark_color_noise
stress fold2 compound_rotation_color_noise


suite,compound,original,single
procedure,,,
baseline,0.955820,0.983375,0.985600
fullframe,0.928351,0.970789,0.968372
fullframe_blend25,0.955788,0.984090,0.987600


## 5. เทียบสอง folds และบันทึกผล
รายงานผลแต่ละ fold แยกจากค่าเฉลี่ย เพื่อไม่ให้ fold ที่ดีบดบัง fold ที่แย่. ยังไม่เรียกเกณฑ์ selection สำหรับ 5 folds และไม่ใช้ audit

In [6]:
parts=[pd.read_parquet(OUT/f'development_fold{k}_scores.parquet') for k in (1,2)]
combined=pd.concat(parts,ignore_index=True)
assert set(combined.fold)=={1,2} and combined.phase.eq('development').all()
assert not combined.duplicated(['fold','condition','procedure']).any()
combined.to_parquet(OUT/'two_fold_scores.parquet',index=False)
display(evaluation.summarize(combined))
s=combined.copy()
s['family']=s.condition.map(lambda c:'original' if c=='original' else ('compound' if c.startswith('compound_') else CONDITIONS[c][1]))
f=s.groupby(['procedure','fold','family']).macro_F1.mean()
rows=[]
for name in ['fullframe','fullframe_blend25']:
    delta=(f.loc[name]-f.loc['baseline']).unstack('family')
    for fold in (1,2):
        rows.append({'procedure':name,'fold':fold,'original_gain':float(delta.loc[fold,'original']),
                     'stress_family_gain':float(delta.drop(columns='original').loc[fold].mean())})
gains=pd.DataFrame(rows)
display(gains)
gains.to_parquet(OUT/'two_fold_gains.parquet',index=False)
condition_scores=combined.pivot(index=['fold','condition'],columns='procedure',values='macro_F1')
display(condition_scores.assign(blend_gain=condition_scores.fullframe_blend25-condition_scores.baseline))
validation.write_json(OUT/'two_fold_report.json',{
    'folds':[1,2],'gains':rows,'test_used':False,'automatic_continuation':False,
    'note':'Exploratory two-fold comparison; not 5-fold confirmation or Public-score prediction'})
print('STOP: two folds complete. Folds 3–5, full refit and Test have not been started.')

suite,compound,original,single
procedure,,,
baseline,0.950273,0.984853,0.981401
fullframe,0.921990,0.972583,0.958758
fullframe_blend25,0.949767,0.985285,0.982789


,procedure,fold,original_gain,stress_family_gain
0,fullframe,1,-0.011954,-0.028230
1,fullframe,2,-0.012586,-0.018935
2,fullframe_blend25,1,0.000149,0.000484
3,fullframe_blend25,2,0.000714,0.001661


procedure                           baseline  fullframe  fullframe_blend25  \
fold condition                                                               
1    blur_1                         0.982007   0.956038           0.979976   
     blur_2                         0.976005   0.897169           0.975972   
     compound_bright_color          0.947935   0.919854           0.949965   
     compound_color_noise           0.949996   0.933860           0.948072   
     compound_dark_color_noise      0.927962   0.891893           0.931979   
     compound_dark_noise            0.954018   0.915592           0.949905   
     compound_rotation_color_noise  0.926192   0.906238           0.922190   
     compound_rotation_dark         0.962249   0.926342           0.960369   
     dark_1                         0.982041   0.964063           0.979998   
     dark_2                         0.976084   0.940079           0.978070   
     gaussian_1                     0.977994   0.954002           0.977979   
     gaussian_2                     0.969964   0.940036           0.977986   
     jpeg_1                         0.973924   0.957986           0.973863   
     jpeg_2                         0.977957   0.954072           0.975937   
     original                       0.986331   0.974377           0.986480   
     speckle_1                      0.980017   0.967993           0.979957   
     speckle_2                      0.976023   0.960003           0.980043   
2    blur_1                         0.987989   0.986018           0.993990   
     blur_2                         0.982014   0.917914           0.982014   
     compound_bright_color          0.960388   0.930189           0.956436   
     compound_color_noise           0.964076   0.952093           0.968023   
     compound_dark_color_noise      0.942069   0.906283           0.946105   
     compound_dark_noise            0.950065   0.923846           0.948014   
     compound_rotation_color_noise  0.942321   0.928014           0.944193   
     compound_rotation_dark         0.976000   0.929681           0.971958   
     dark_1                         0.985968   0.975972           0.987999   
     dark_2                         0.984027   0.956036           0.984027   
     gaussian_1                     0.986009   0.981987           0.990009   
     gaussian_2                     0.982009   0.953776           0.981999   
     jpeg_1                         0.989990   0.983988           0.991980   
     jpeg_2                         0.985999   0.983999           0.991990   
     original                       0.983375   0.970789           0.984090   
     speckle_1                      0.992000   0.977998           0.992000   
     speckle_2                      0.979998   0.966036           0.979998   

procedure                             blend_gain  
fold condition                                    
1    blur_1                        -2.031231e-03  
     blur_2                        -3.263037e-05  
     compound_bright_color          2.030251e-03  
     compound_color_noise          -1.923832e-03  
     compound_dark_color_noise      4.017790e-03  
     compound_dark_noise           -4.113631e-03  
     compound_rotation_color_noise -4.002644e-03  
     compound_rotation_dark        -1.879654e-03  
     dark_1                        -2.043551e-03  
     dark_2                         1.985587e-03  
     gaussian_1                    -1.460607e-05  
     gaussian_2                     8.022843e-03  
     jpeg_1                        -6.058221e-05  
     jpeg_2                        -2.019591e-03  
     original                       1.488898e-04  
     speckle_1                     -6.002861e-05  
     speckle_2                      4.019727e-03  
2    blur_1                         6.000349e-03  
     blur_2                        -4.485561e-07  
     compound_bright_color         -3.951673e-03  
     compound_color_noise           3.947415e-03  
     compound_dark

STOP: two folds complete. Folds 3–5, full refit and Test have not been started.
